## Optimizer Comparision - Performance Tests

In [1]:
! pip install -q neptune torch-adopt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.9/487.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:0

In [ ]:
import os

NEPTUNE_PROJECT_NAME = "sigk/optimizer-comparision"
NEPTUNE_API_KEY = input("Enter Neptune API key: ")

In [5]:
import neptune
from torch import nn
from torch import optim
from torch.utils import data

def save_optimizer_config(run: neptune.Run, optimizer: optim.Optimizer):
    if run is None:
        return

    run["optimizer/type"] = optimizer.__class__.__name__

    optimizer_config = {}

    if hasattr(optimizer, 'defaults'):
        for key, value in optimizer.defaults.items():
            if isinstance(value, (int, float, str, bool)):
                optimizer_config[key] = value
            elif isinstance(value, (list, tuple)):
                optimizer_config[key] = list(value)
            else:
                optimizer_config[key] = str(value)

    for i, param_group in enumerate(optimizer.param_groups):
        group_config = {}
        for key, value in param_group.items():
            if key != 'params':
                if isinstance(value, (int, float, str, bool)):
                    group_config[key] = value
                elif isinstance(value, (list, tuple)):
                    group_config[key] = list(value)
                else:
                    group_config[key] = str(value)

        for key, value in group_config.items():
            run[f"optimizer/param_group_{i}/{key}"] = value if isinstance(value, (int, float, str, bool)) else str(value)

    for key, value in optimizer_config.items():
        run[f"optimizer/config/{key}"] = value if isinstance(value, (int, float, str, bool)) else str(value)


def save_run_data(
    run: neptune.Run,
    task: str,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    criterion: nn.Module,
    epochs: int,
    device: str,
):
    run["model"] = model.__class__.__name__
    run["criterion"] = criterion.__class__.__name__
    run["task"] = task
    run["epochs"] = epochs
    run["batch_size"] = train_loader.batch_size
    run["device"] = device
    save_optimizer_config(run, optimizer)

### CIFAR100 Image Classification

In [3]:
from typing import Tuple

import torch
import torchvision
from torch import nn
from torch import optim
from torch.utils import data
from torchvision import models, transforms, datasets

CIFAR100_ROOT = "/data/cifar100"


def cifar100_model_factory(num_classes: int = 100) -> nn.Module:
    model = models.resnet18(num_classes=num_classes)
    # modify first conv layer to avoid upscaling to 224x224
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def cifar100_dataloader_factory(
    batch_size: int = 32,
    seed: int = 42
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    # standard CIFAR-100 mean and std
    mean = [0.5071, 0.4865, 0.4409]
    std = [0.2673, 0.2564, 0.2761]

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    temp_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=True, transform=transform, download=True)
    test_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=False, transform=transform, download=True)

    train_size = int(0.8 * len(temp_dataset))
    val_size = len(temp_dataset) - train_size
    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = data.random_split(temp_dataset, [train_size, val_size], generator=generator)

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [17]:
from datetime import datetime

import neptune
from sklearn.metrics import accuracy_score, f1_score

def cifar100_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 10,
    run: neptune.Run | None = None,
    device: str = "cuda",
) -> float:
    model = model.to(device)
    total_train_time = 0.0
    total_val_time = 0.0

    for epoch in range(epochs):
        if run:
            run["train/epoch/lr"].append(optimizer.param_groups[0]['lr'])

        train_start = datetime.now()
        model.train()
        epoch_grad_norms = []
        train_preds = []
        train_targets = []
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().tolist())
            train_targets.extend(targets.cpu().tolist())

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf'))
            epoch_grad_norms.append(grad_norm.item())
            optimizer.step()


            if run:
                run["train/step/loss"].append(loss.item())
                run["train/step/grad_norm"].append(grad_norm)

        train_end = datetime.now()
        epoch_train_time = (train_end - train_start).total_seconds()
        total_train_time += epoch_train_time

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_targets, train_preds, average="macro")
        train_accuracy = accuracy_score(train_targets, train_preds)
        avg_grad_norm = sum(epoch_grad_norms) / len(epoch_grad_norms)
        if run:
            run["train/epoch/time"].append(epoch_train_time)
            run["train/epoch/loss"].append(train_loss)
            run["train/epoch/f1"].append(train_f1)
            run["train/epoch/accuracy"].append(train_accuracy)
            run["train/epoch/grad_norm"].append(avg_grad_norm)

        val_start = datetime.now()
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().tolist())
                val_targets.extend(targets.cpu().tolist())

                if run:
                    run["val/step/loss"].append(loss.item())

        val_end = datetime.now()
        epoch_val_time = (val_end - val_start).total_seconds()
        total_val_time += epoch_val_time

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        val_accuracy = accuracy_score(val_targets, val_preds)
        if run:
            run["val/epoch/time"].append(epoch_val_time)
            run["val/epoch/loss"].append(val_loss)
            run["val/epoch/f1"].append(val_f1)
            run["val/epoch/accuracy"].append(val_accuracy)


        print(f"Epoch {epoch+1}/{epochs} — F1 Score: {val_f1:.4f} | Accuracy: {val_accuracy:.4f} | Val Loss: {val_loss:.4f} | Train Loss {train_loss:.4f}")

    print(f"Total training time: {total_train_time:.2f}s")
    print(f"Total validation time: {total_val_time:.2f}s")
    print(f"Total time: {total_train_time + total_val_time:.2f}s")

    if run:
        run["train/total_time"].append(total_train_time)
        run["val/total_time"].append(total_val_time)
        run["total_time"].append(total_train_time + total_val_time)


In [19]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report

def cifar100_test(
    *,
    model: nn.Module,
    test_loader: data.DataLoader,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    run: neptune.Run | None = None,
    device: str = "cuda",
) -> dict:
    model = model.to(device)
    model.eval()

    test_start = datetime.now()

    all_preds = []
    all_targets = []
    test_loss = 0.0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())

            if run:
                run["test/step/loss"].append(loss.item())

    test_end = datetime.now()
    test_time = (test_end - test_start).total_seconds()

    test_loss /= len(test_loader)
    test_accuracy = accuracy_score(all_targets, all_preds)
    test_f1_macro = f1_score(all_targets, all_preds, average="macro")
    test_f1_micro = f1_score(all_targets, all_preds, average="micro")
    test_f1_weighted = f1_score(all_targets, all_preds, average="weighted")
    test_precision_macro = precision_score(all_targets, all_preds, average="macro")
    test_recall_macro = recall_score(all_targets, all_preds, average="macro")

    if run:
        run["test/loss"] = test_loss
        run["test/accuracy"] = test_accuracy
        run["test/f1_macro"] = test_f1_macro
        run["test/f1_micro"] = test_f1_micro
        run["test/f1_weighted"] = test_f1_weighted
        run["test/precision_macro"] = test_precision_macro
        run["test/recall_macro"] = test_recall_macro
        run["test/time"] = test_time
        run["test/num_samples"] = len(all_targets)

        class_report = classification_report(all_targets, all_preds,
                                           target_names=[f"class_{i}" for i in range(100)])
        run["test/classification_report"] = class_report

    print(f"\n=== TEST RESULTS ===")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test F1 (Macro): {test_f1_macro:.4f}")
    print(f"Test F1 (Micro): {test_f1_micro:.4f}")
    print(f"Test F1 (Weighted): {test_f1_weighted:.4f}")
    print(f"Test Precision (Macro): {test_precision_macro:.4f}")
    print(f"Test Recall (Macro): {test_recall_macro:.4f}")
    print(f"Test Time: {test_time:.2f}s")
    print(f"Samples Tested: {len(all_targets)}")

#### Test the generators and tracking functions

In [12]:
model = cifar100_model_factory()
optimizer = optim.Adam(model.parameters())

In [14]:
train_loader, val_loader, test_loader = cifar100_dataloader_factory()

100%|██████████| 169M/169M [00:12<00:00, 13.2MB/s]


In [13]:
run = neptune.init_run(
    project=NEPTUNE_PROJECT_NAME,
    api_token=NEPTUNE_API_KEY,
)

[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/sigk/optimizer-comparision/e/OP-1


In [15]:
cifar100_train_eval_loop(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    run=run,
)

[neptune] [warning] NeptuneUnsupportedType: You're attempting to log a type that is not directly supported by Neptune (<class 'list'>).
        Convert the value to a supported type, such as a string or float, or use stringify_unsupported(obj)
        for dictionaries or collections that contain unsupported values.
        For more, see https://docs-legacy.neptune.ai/help/value_of_unsupported_type


Epoch 1/10 — F1 Score: 0.1677 | Accuracy: 0.1979 | Val Loss: 3.2279 | Train Loss 3.7821
Epoch 2/10 — F1 Score: 0.2826 | Accuracy: 0.3024 | Val Loss: 2.7099 | Train Loss 2.9146
Epoch 3/10 — F1 Score: 0.3863 | Accuracy: 0.3998 | Val Loss: 2.2591 | Train Loss 2.3178
Epoch 4/10 — F1 Score: 0.4509 | Accuracy: 0.4586 | Val Loss: 2.0055 | Train Loss 1.8839
Epoch 5/10 — F1 Score: 0.4859 | Accuracy: 0.4937 | Val Loss: 1.8532 | Train Loss 1.5106
Epoch 6/10 — F1 Score: 0.4941 | Accuracy: 0.4952 | Val Loss: 1.9585 | Train Loss 1.1568
Epoch 7/10 — F1 Score: 0.5131 | Accuracy: 0.5160 | Val Loss: 1.9419 | Train Loss 0.8007
Epoch 8/10 — F1 Score: 0.4915 | Accuracy: 0.4987 | Val Loss: 2.2578 | Train Loss 0.4921
Epoch 9/10 — F1 Score: 0.5048 | Accuracy: 0.5034 | Val Loss: 2.4646 | Train Loss 0.3030
Epoch 10/10 — F1 Score: 0.4986 | Accuracy: 0.4955 | Val Loss: 2.6995 | Train Loss 0.2174
Total training time: 517.56s
Total validation time: 49.59s
Total time: 567.15s


In [21]:
cifar100_test(
    model=model,
    test_loader=test_loader,
    run=run,
)


=== TEST RESULTS ===
Test Loss: 2.7068
Test Accuracy: 0.5007
Test F1 (Macro): 0.5053
Test F1 (Micro): 0.5007
Test F1 (Weighted): 0.5053
Test Precision (Macro): 0.5459
Test Recall (Macro): 0.5007
Test Time: 6.78s
Samples Tested: 10000


In [22]:
run.stop()

[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] All 0 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/sigk/optimizer-comparision/e/OP-1/metadata


#### Full run function

In [23]:
def cifar100_run_for_optimizer(optimizer_class: nn.Module, **optimizer_kwargs):
    run = neptune.init_run(
        project=NEPTUNE_PROJECT_NAME,
        api_token=NEPTUNE_API_KEY,
    )

    task = "image-classification-cifar100"
    model = cifar100_model_factory()
    optimizer = optimizer_class(model.parameters(), **optimizer_kwargs)
    train_loader, val_loader, test_loader = cifar100_dataloader_factory()

    criterion=nn.CrossEntropyLoss()
    epochs = 10
    device = "cuda" if torch.cuda.is_available() else "cpu"

    save_run_data(
        run=run,
        task=task,
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        epochs=epochs,
        device=device,
    )

    cifar100_train_eval_loop(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        epochs=epochs,
        run=run,
        device=device,
    )

    cifar100_test(
        model=model,
        test_loader=test_loader,
        criterion=criterion,
        run=run,
        device=device,
    )

    run.stop()

### Regression on Real Estate Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils import data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import re
from datetime import datetime


class RealEstateNN(nn.Module):
    def __init__(self, input_size):
        super(RealEstateNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


def realestate_model_factory(input_size: int) -> nn.Module:
    return RealEstateNN(input_size)


def realestate_preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data = data.dropna(subset=['Sale Amount'])

    selected_features = [
        'Assessed Value',
        'Property Type',
        'Residential Type',
        'Town',
        'Date Recorded'
    ]


    data['Latitude'] = data['Location'].str.extract(r'POINT \(([+-]?\d*\.?\d+)\s+([+-]?\d*\.?\d+)\)')[1].astype(float)
    data['Longitude'] = data['Location'].str.extract(r'POINT \(([+-]?\d*\.?\d+)\s+([+-]?\d*\.?\d+)\)')[0].astype(float)
    selected_features.extend(['Latitude', 'Longitude'])

    features_df = data[selected_features + ['Sale Amount']].copy()

    features_df['Date Recorded'] = pd.to_datetime(features_df['Date Recorded'], errors='coerce')
    features_df['Year'] = features_df['Date Recorded'].dt.year
    features_df['Month'] = features_df['Date Recorded'].dt.month
    features_df['Day_of_Year'] = features_df['Date Recorded'].dt.dayofyear
    features_df = features_df.drop('Date Recorded', axis=1)

    numerical_cols = ['Assessed Value', 'Latitude', 'Longitude', 'Year', 'Month', 'Day_of_Year']
    for col in numerical_cols:
        if col in features_df.columns:
            features_df[col] = features_df[col].fillna(features_df[col].median())

    categorical_cols = ['Property Type', 'Residential Type', 'Town']
    for col in categorical_cols:
        if col in features_df.columns:
            features_df[col] = features_df[col].fillna('Unknown')

    label_encoders = {}
    for col in categorical_cols:
        if col in features_df.columns:
            le = LabelEncoder()
            features_df[col] = le.fit_transform(features_df[col].astype(str))
            label_encoders[col] = le

    X = features_df.drop('Sale Amount', axis=1)
    y = features_df['Sale Amount']

    mask = ~(X.isnull().any(axis=1) | y.isnull())
    X = X[mask]
    y = y[mask]

    return X, y, label_encoders


def realestate_dataloader_factory(
    dataset_path: str,
    batch_size: int = 64,
    test_size: float = 0.2,
    val_size: float = 0.2
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    df = pd.read_csv(dataset_path)

    X, y, label_encoders = realestate_preprocess_data(df)

    X_np = X.values.astype(np.float32)
    y_np = y.values.astype(np.float32)

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_np, y_np, test_size=test_size, random_state=42, stratify=None
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size, random_state=42, stratify=None
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    train_dataset = data.TensorDataset(
        torch.FloatTensor(X_train_scaled),
        torch.FloatTensor(y_train)
    )
    val_dataset = data.TensorDataset(
        torch.FloatTensor(X_val_scaled),
        torch.FloatTensor(y_val)
    )
    test_dataset = data.TensorDataset(
        torch.FloatTensor(X_test_scaled),
        torch.FloatTensor(y_test)
    )

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    input_size = X_train_scaled.shape[1]
    print(f"Data input size: {input_size}")

    return train_loader, val_loader, test_loader

In [24]:
from datetime import datetime

import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score


def realestate_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    criterion: nn.Module = nn.MSELoss(),
    epochs: int = 50,
    run: neptune.Run | None = None,
    device: str = "cuda",
) -> float:
    model = model.to(device)
    total_train_time = 0.0
    total_val_time = 0.0

    for epoch in range(epochs):
        if run:
            run["train/epoch/lr"].append(optimizer.param_groups[0]['lr'])

        train_start = datetime.now()
        model.train()
        epoch_grad_norms = []
        train_loss = 0.0
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            targets = targets.view(-1, 1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            train_loss += loss.item()

            train_preds.extend(outputs.cpu().detach().numpy().flatten())
            train_targets.extend(targets.cpu().numpy().flatten())

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf'))
            epoch_grad_norms.append(grad_norm.item())
            optimizer.step()

            if run:
                run["train/step/loss"].append(loss.item())
                run["train/step/grad_norm"].append(grad_norm)

        train_end = datetime.now()
        epoch_train_time = (train_end - train_start).total_seconds()
        total_train_time += epoch_train_time

        train_loss /= len(train_loader)
        train_mae = mean_absolute_error(train_targets, train_preds)
        train_rmse = np.sqrt(np.mean((np.array(train_targets) - np.array(train_preds))**2))
        train_r2 = r2_score(train_targets, train_preds)
        avg_grad_norm = sum(epoch_grad_norms) / len(epoch_grad_norms)
        if run:
            run["train/epoch/time"].append(epoch_train_time)
            run["train/epoch/loss"].append(train_loss)
            run["train/epoch/mae"].append(train_mae)
            run["train/epoch/rmse"].append(train_rmse)
            run["train/epoch/r2"].append(train_r2)
            run["train/epoch/grad_norm"].append(avg_grad_norm)

        val_start = datetime.now()
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                targets = targets.view(-1, 1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                val_preds.extend(outputs.cpu().numpy().flatten())
                val_targets.extend(targets.cpu().numpy().flatten())

                if run:
                    run["val/step/loss"].append(loss.item())

        val_end = datetime.now()
        epoch_val_time = (val_end - val_start).total_seconds()
        total_val_time += epoch_val_time

        val_loss /= len(val_loader)
        val_mae = mean_absolute_error(val_targets, val_preds)
        val_rmse = np.sqrt(np.mean((np.array(val_targets) - np.array(val_preds))**2))
        val_r2 = r2_score(val_targets, val_preds)

        if run:
            run["val/epoch/time"].append(epoch_val_time)
            run["val/epoch/loss"].append(val_loss)
            run["val/epoch/mae"].append(val_mae)
            run["val/epoch/rmse"].append(val_rmse)
            run["val/epoch/r2"].append(val_r2)

        print(f"Epoch {epoch+1}/{epochs} — Val Loss: {val_loss:.4f} | Train Loss {train_loss:.4f}")

    print(f"Total training time: {total_train_time:.2f}s")
    print(f"Total validation time: {total_val_time:.2f}s")
    print(f"Total time: {total_train_time + total_val_time:.2f}s")

    if run:
        run["train/total_time"].append(total_train_time)
        run["val/total_time"].append(total_val_time)
        run["total_time"].append(total_train_time + total_val_time)
